# Train YOLO troop detector on Colab

Dataset: https://github.com/wty-yy/Clash-Royale-Detection-Dataset (MIT)

**Before running:** `Runtime` â†’ `Change runtime type` â†’ **T4 GPU**.

Cells in order. Everything stays in `/content/`; final cell downloads `best.pt` directly to your machine. Don't close the tab â€” if the runtime dies, you start over.

## 1. Sanity check + install

In [ ]:
!nvidia-smi
!pip install -q ultralytics pyyaml

## 2. Clone the dataset

In [ ]:
!git clone --depth 1 https://github.com/wty-yy/Clash-Royale-Detection-Dataset.git /content/cr-data
!wget -q https://raw.githubusercontent.com/wty-yy/KataCR/master/katacr/constants/label_list.py -O /content/cr-data/label_list.py
!ls /content/cr-data/images/part2 | head

## 3. Convert labels + build YOLO tree

The dataset's labels are non-standard YOLO: `cls x y w h belonging 0 0 0 0 0 0`. The `belonging` field encodes ally/enemy on a separate axis. Standard YOLO can't predict a second axis, so we **fold team into the class**: each original class `k` becomes two classes â€” `2k` (ally) and `2k+1` (enemy). This matches the `_parse_team` logic in `detector.py`, which looks for `ally`/`enemy` in class names.

If teams come out flipped after the first training run, swap `TEAM_BY_BEL` and retrain.

In [ ]:
import os, random, shutil, importlib.util, yaml
from pathlib import Path

DATA_REPO   = Path('/content/cr-data')
SRC_ROOT    = DATA_REPO / 'images' / 'part2'
OUT_ROOT    = Path('/content/dataset')
TRAIN_RATIO = 0.8
SEED        = 42
TEAM_BY_BEL = ['ally', 'enemy']  # bel 0 -> ally, bel 1 -> enemy. Flip if swapped.

spec = importlib.util.spec_from_file_location('label_list', DATA_REPO / 'label_list.py')
m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
class_names = next(
    v for v in vars(m).values()
    if isinstance(v, list) and v and all(isinstance(x, str) for x in v)
)
print(f'original classes: {len(class_names)}')

folded = []
for n in class_names:
    folded.append(f'{n}_{TEAM_BY_BEL[0]}')
    folded.append(f'{n}_{TEAM_BY_BEL[1]}')
print(f'folded classes: {len(folded)}')

pairs = []
for txt in SRC_ROOT.rglob('*.txt'):
    for ext in ('.png', '.jpg', '.jpeg'):
        img = txt.with_suffix(ext)
        if img.exists():
            pairs.append((img, txt))
            break
print(f'image+label pairs: {len(pairs)}')

random.seed(SEED); random.shuffle(pairs)
split = int(len(pairs) * TRAIN_RATIO)
splits = {'train': pairs[:split], 'val': pairs[split:]}

for name, items in splits.items():
    img_dir = OUT_ROOT / 'images' / name; img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir = OUT_ROOT / 'labels' / name; lbl_dir.mkdir(parents=True, exist_ok=True)
    for img, txt in items:
        flat = '__'.join(img.relative_to(SRC_ROOT).with_suffix('').parts)
        shutil.copy2(img, img_dir / f'{flat}{img.suffix}')
        out_lines = []
        for line in txt.read_text().splitlines():
            p = line.strip().split()
            if len(p) < 6:
                continue
            cls, x, y, w, h, bel = int(p[0]), p[1], p[2], p[3], p[4], int(p[5])
            out_lines.append(f'{cls*2 + bel} {x} {y} {w} {h}')
        (lbl_dir / f'{flat}.txt').write_text('\n'.join(out_lines))
    print(f'  {name}: {len(items)} images')

(OUT_ROOT / 'data.yaml').write_text(yaml.safe_dump({
    'path':  str(OUT_ROOT),
    'train': 'images/train',
    'val':   'images/val',
    'names': folded,
}, sort_keys=False))
print(f'wrote {OUT_ROOT}/data.yaml')

## 4. Train

YOLOv8s pretrained on COCO; only the head is fresh (class space is different). ~2-4h on a T4 for 100 epochs. `patience=20` early-stops if val mAP plateaus.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='run1',
    exist_ok=True,
    patience=20,
)

## 5. Download `best.pt`
Triggers a browser download. Drop the file into `models/best.pt` in the local repo.

In [ ]:
from google.colab import files
files.download('/content/runs/run1/weights/best.pt')